In [1]:
import pandas as pd
import numpy as np

from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Labs"
)

xpt_file = DATA_DIR / "TRIGLY_G.xpt"
csv_file = DATA_DIR / "TRIGLY_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 80)
print("NHANES 2011-2012 TRIGLY_G XPT -> CSV")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WTSAF2YR",
    "LBXTR",
    "LBDTRSI",
    "LBDLDL",
    "LBDLDLSI"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "TRIGLY_G column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 3239:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 6:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 3,239 rows x 6 columns."
)


# ============================================================
# 4. DEFINE VARIABLE TYPES
# ============================================================

#
# Genuine decimal variables:
#
# WTSAF2YR
#     fasting subsample survey weight
#
# LBDTRSI
#     triglycerides mmol/L
#
# LBDLDLSI
#     LDL cholesterol mmol/L
#
# These MUST NOT be converted to Int64.
#

DECIMAL_COLUMNS = [
    "WTSAF2YR",
    "LBDTRSI",
    "LBDLDLSI"
]


#
# Proven whole-number fields:
#
# SEQN
# LBXTR
# LBDLDL
#

INTEGER_COLUMNS = [
    "SEQN",
    "LBXTR",
    "LBDLDL"
]


print("\n--- VARIABLE TYPE PLAN ---")

print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number variables:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# NEVER perform a blanket replacement before validating
# the exact variables and counts.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)

print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Attached TRIGLY_G.xpt:
#
# WTSAF2YR = 397
#
# Total = 397
#
# CDC publishes WTSAF2YR range beginning at 0,
# so these are validated zero sample weights.
#

EXPECTED_TINY_COUNTS = {
    "WTSAF2YR": 397
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated TRIGLY_G pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 397:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


print(
    "\nPASS: Exact WTSAF2YR 397-value "
    "tiny-zero pattern validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED WTSAF2YR ZEROS
# ============================================================

#
# Only WTSAF2YR tiny values are changed.
#
# No global replacement.
#

tiny_mask = (
    df["WTSAF2YR"]
    .eq(TINY_VALUE)
)


correction_count = int(
    tiny_mask.sum()
)


if correction_count != 397:

    raise ValueError(
        "Unexpected WTSAF2YR tiny-value count."
    )


df.loc[
    tiny_mask,
    "WTSAF2YR"
] = 0


print("\n--- ZERO RESTORATION ---")

print(
    "WTSAF2YR restored to zero:",
    correction_count
)


if int(
    df["WTSAF2YR"]
    .eq(0)
    .sum()
) != 397:

    raise ValueError(
        "WTSAF2YR zero restoration failed."
    )


print(
    "PASS: 397 validated WTSAF2YR zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df differs from raw pandas XPT ONLY by:
#
#     WTSAF2YR tiny -> 0
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["WTSAF2YR"]
    .eq(TINY_VALUE),
    "WTSAF2YR"
] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values after correction:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Critical fidelity check BEFORE Int64 conversion.
#
# Expected decimal variables:
#
# WTSAF2YR
# LBDTRSI
# LBDLDLSI
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{count:,} fractional observations"
    )


    if count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "Unexpected decimal-variable pattern.\n"
        f"Expected: {DECIMAL_COLUMNS}\n"
        f"Actual:   {fractional_columns}"
    )


print(
    "PASS: Exact decimal-variable pattern confirmed."
)


# ============================================================
# 11. VALIDATE WTSAF2YR
# ============================================================

#
# CDC:
#
# WTSAF2YR range begins at 0.
# Count = 3,239
# Missing = 0
#
# Attached XPT after correction:
#
# zeros = 397
#
# IMPORTANT:
#
# WTSAF2YR is a genuine decimal sampling weight.
# Do NOT convert it to Int64.
#

s = df["WTSAF2YR"]


print("\n--- WTSAF2YR ---")

print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)

print(
    "Zeros:",
    int(
        s.eq(0)
        .sum()
    )
)

print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)

print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    np.isclose(
        s.min(),
        0,
        rtol=0,
        atol=0
    )
    and
    int(
        s.notna()
        .sum()
    ) == 3239
    and
    int(
        s.isna()
        .sum()
    ) == 0
    and
    int(
        s.eq(0)
        .sum()
    ) == 397
):

    raise ValueError(
        "WTSAF2YR validation failed."
    )


#
# CDC webpage prints the maximum as approximately
# 521033.37059.
#
# The attached XPT decodes as approximately:
#
# 521033.371
#
# DO NOT modify the XPT value merely to force the
# webpage display precision.
#

if not np.isclose(
    s.max(),
    521033.37059,
    rtol=0,
    atol=0.001
):

    raise ValueError(
        "WTSAF2YR maximum unexpectedly differs "
        "from CDC range."
    )


print(
    "PASS: WTSAF2YR matches CDC range/count."
)


# ============================================================
# 12. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 13. FORMAT WHOLE-NUMBER LAB RESULTS
# ============================================================

#
# LBXTR:
#     triglycerides mg/dL
#
# LBDLDL:
#     calculated LDL mg/dL
#
# Both are released as whole-number values.
#
# NO .round() is used.
#

for col in [
    "LBXTR",
    "LBDLDL"
]:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if not np.all(
        np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )
    ):

        raise ValueError(
            f"{col} unexpectedly contains "
            "fractional values."
        )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: LBXTR and LBDLDL converted "
    "to Int64 without rounding."
)


# ============================================================
# 14. CDC CHECK - LBXTR
# ============================================================

#
# CDC:
#
# LBXTR
# Triglyceride (mg/dL)
#
# Range = 18-1562
# Valid = 2,990
# Missing = 249
#

s = df["LBXTR"]


print("\n--- LBXTR ---")

print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)

print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)

print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    s.min() == 18
    and
    s.max() == 1562
    and
    int(
        s.notna()
        .sum()
    ) == 2990
    and
    int(
        s.isna()
        .sum()
    ) == 249
):

    raise ValueError(
        "LBXTR differs from CDC."
    )


print(
    "PASS: LBXTR matches CDC."
)


# ============================================================
# 15. CDC CHECK - LBDTRSI
# ============================================================

#
# CDC:
#
# LBDTRSI
# Triglyceride mmol/L
#
# 0.203 - 17.635
# Valid = 2,990
# Missing = 249
#

s = df["LBDTRSI"]


print("\n--- LBDTRSI ---")

print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)

print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)

print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    np.isclose(
        s.min(),
        0.203,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        17.635,
        rtol=0,
        atol=0
    )
    and
    int(
        s.notna()
        .sum()
    ) == 2990
    and
    int(
        s.isna()
        .sum()
    ) == 249
):

    raise ValueError(
        "LBDTRSI differs from CDC."
    )


print(
    "PASS: LBDTRSI matches CDC."
)


# ============================================================
# 16. VERIFY TRIGLYCERIDE SI CONVERSION
# ============================================================

#
# CDC formula:
#
#     LBDTRSI = LBXTR × 0.01129
#
# Released to three decimal places.
#
# Use Decimal + ROUND_HALF_UP.
#

def trig_to_si(
    value
):

    result = (
        Decimal(
            str(int(value))
        )
        *
        Decimal("0.01129")
    )


    return float(
        result.quantize(
            Decimal("0.001"),
            rounding=ROUND_HALF_UP
        )
    )


trig_pairs = (
    df[
        [
            "LBXTR",
            "LBDTRSI"
        ]
    ]
    .dropna()
)


expected_trig_si = (
    trig_pairs["LBXTR"]
    .apply(
        trig_to_si
    )
    .to_numpy(
        dtype=float
    )
)


actual_trig_si = (
    trig_pairs["LBDTRSI"]
    .to_numpy(
        dtype=float
    )
)


trig_si_mismatches = int(
    np.sum(
        expected_trig_si
        !=
        actual_trig_si
    )
)


print("\n--- TRIGLYCERIDE SI CHECK ---")

print(
    "Pairs checked:",
    f"{len(trig_pairs):,}"
)

print(
    "Mismatches:",
    trig_si_mismatches
)


if trig_si_mismatches != 0:

    raise ValueError(
        "LBDTRSI does not match "
        "LBXTR × 0.01129."
    )


print(
    "PASS: All 2,990 triglyceride SI "
    "values match CDC conversion."
)


# ============================================================
# 17. CDC CHECK - LBDLDL
# ============================================================

#
# CDC:
#
# LBDLDL
#
# 9 - 331 mg/dL
# Valid = 2,942
# Missing = 297
#

s = df["LBDLDL"]


print("\n--- LBDLDL ---")

print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)

print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)

print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    s.min() == 9
    and
    s.max() == 331
    and
    int(
        s.notna()
        .sum()
    ) == 2942
    and
    int(
        s.isna()
        .sum()
    ) == 297
):

    raise ValueError(
        "LBDLDL differs from CDC."
    )


print(
    "PASS: LBDLDL matches CDC."
)


# ============================================================
# 18. CHECK LDL AVAILABILITY VS TRIGLYCERIDES
# ============================================================

#
# CDC Friedewald calculation is valid only when:
#
#     triglycerides <= 400 mg/dL
#
# Attached file:
#
# LBXTR <= 400 = 2,942
# LBXTR > 400  = 48
# LBXTR missing = 249
#
# LBDLDL:
#
# valid   = 2,942
# missing = 297
#
# 249 + 48 = 297
#

trig_eligible = (
    df["LBXTR"]
    .notna()
    &
    df["LBXTR"]
    .le(400)
)


trig_high = (
    df["LBXTR"]
    .gt(400)
)


print("\n--- LDL ELIGIBILITY CHECK ---")

print(
    "Triglycerides <=400:",
    int(
        trig_eligible.sum()
    )
)

print(
    "Triglycerides >400:",
    int(
        trig_high.sum()
    )
)


if int(
    trig_eligible.sum()
) != 2942:

    raise ValueError(
        "Unexpected triglyceride <=400 count."
    )


if int(
    trig_high.sum()
) != 48:

    raise ValueError(
        "Unexpected triglyceride >400 count."
    )


if not (
    df.loc[
        trig_eligible,
        "LBDLDL"
    ]
    .notna()
    .all()
):

    raise ValueError(
        "Some eligible LDL records are missing."
    )


if not (
    df.loc[
        trig_high,
        "LBDLDL"
    ]
    .isna()
    .all()
):

    raise ValueError(
        "LDL is present where triglycerides >400."
    )


print(
    "PASS: LDL availability follows CDC "
    "Friedewald eligibility rule."
)


# ============================================================
# 19. CDC CHECK - LBDLDLSI
# ============================================================

#
# CDC:
#
# LBDLDLSI
#
# 0.233 - 8.56 mmol/L
# Valid = 2,942
# Missing = 297
#

s = df["LBDLDLSI"]


print("\n--- LBDLDLSI ---")

print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)

print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)

print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    np.isclose(
        s.min(),
        0.233,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        8.56,
        rtol=0,
        atol=0
    )
    and
    int(
        s.notna()
        .sum()
    ) == 2942
    and
    int(
        s.isna()
        .sum()
    ) == 297
):

    raise ValueError(
        "LBDLDLSI differs from CDC."
    )


print(
    "PASS: LBDLDLSI matches CDC."
)


# ============================================================
# 20. VERIFY LDL SI CONVERSION
# ============================================================

#
# CDC:
#
#     LBDLDLSI = LBDLDL × 0.02586
#
# Released to three decimal places.
#

def ldl_to_si(
    value
):

    result = (
        Decimal(
            str(int(value))
        )
        *
        Decimal("0.02586")
    )


    return float(
        result.quantize(
            Decimal("0.001"),
            rounding=ROUND_HALF_UP
        )
    )


ldl_pairs = (
    df[
        [
            "LBDLDL",
            "LBDLDLSI"
        ]
    ]
    .dropna()
)


expected_ldl_si = (
    ldl_pairs["LBDLDL"]
    .apply(
        ldl_to_si
    )
    .to_numpy(
        dtype=float
    )
)


actual_ldl_si = (
    ldl_pairs["LBDLDLSI"]
    .to_numpy(
        dtype=float
    )
)


ldl_si_mismatches = int(
    np.sum(
        expected_ldl_si
        !=
        actual_ldl_si
    )
)


print("\n--- LDL SI CHECK ---")

print(
    "Pairs checked:",
    f"{len(ldl_pairs):,}"
)

print(
    "Mismatches:",
    ldl_si_mismatches
)


if ldl_si_mismatches != 0:

    raise ValueError(
        "LBDLDLSI does not match "
        "LBDLDL × 0.02586."
    )


print(
    "PASS: All 2,942 LDL SI values "
    "match CDC conversion."
)


# ============================================================
# 21. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY intended numeric difference from pandas'
# raw XPT:
#
#     WTSAF2YR:
#         397 TINY_VALUE -> 0
#
# Nothing else may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 6 variables match the "
    "expected corrected XPT exactly."
)


# ============================================================
# 22. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO float_format.
# NO rounding.
#
# Decimal fields remain natural floats.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 23. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 24. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print("\n--- EXACT XPT -> CSV FIDELITY ---")


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    if not np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 6 variables round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 25. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 26. READ RAW CSV AS TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 27. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 28. .0 CHECK - WHOLE-NUMBER VARIABLES ONLY
# ============================================================

#
# IMPORTANT:
#
# DO NOT search the whole CSV for ".0".
#
# WTSAF2YR and LBDLDLSI are genuine decimal fields
# and may legitimately display values ending in ".0".
#
# Check only:
#
#     SEQN
#     LBXTR
#     LBDLDL
#

print("\n--- INTEGER .0 FORMAT CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting in "
        "whole-number variables:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 in "
    "SEQN/LBXTR/LBDLDL."
)


# ============================================================
# 29. EXACT DECIMAL FIDELITY
# ============================================================

print("\n--- DECIMAL VALUE FIDELITY ---")


decimal_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    if not np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        decimal_failures.append(
            col
        )


if decimal_failures:

    raise ValueError(
        "Decimal fidelity failed:\n"
        + str(decimal_failures)
    )


print(
    "PASS: WTSAF2YR, LBDTRSI and "
    "LBDLDLSI preserved exactly."
)


# ============================================================
# 30. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 31. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no .0."
)


# ============================================================
# 32. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "TRIGLY_G CONVERSION COMPLETE"
)


print(
    "=" * 80
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "WTSAF2YR zero restorations:",
    correction_count
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "Whole-number variables:",
    INTEGER_COLUMNS
)


print(
    "Triglyceride SI conversion mismatches:",
    trig_si_mismatches
)


print(
    "LDL SI conversion mismatches:",
    ldl_si_mismatches
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary .0 in integer fields: "
    "0 expected"
)


print(
    "Lab decimal values altered: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 TRIGLY_G XPT -> CSV
Rows: 3,239
Columns: 6
PASS: Structure = 3,239 rows x 6 columns.

--- VARIABLE TYPE PLAN ---
Genuine decimal variables: ['WTSAF2YR', 'LBDTRSI', 'LBDLDLSI']
Whole-number variables: ['SEQN', 'LBXTR', 'LBDLDL']

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros: 0
Tiny artifact occurrences: 397

Tiny values by variable:
WTSAF2YR    397

PASS: Exact WTSAF2YR 397-value tiny-zero pattern validated.

--- ZERO RESTORATION ---
WTSAF2YR restored to zero: 397
PASS: 397 validated WTSAF2YR zeros restored.

Tiny values after correction: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
WTSAF2YR: 2,841 fractional observations
LBXTR: 0 fractional observations
LBDTRSI: 2,990 fractional observations
LBDLDL: 0 fractional observations
LBDLDLSI: 2,912 fractional observations
PASS: Exact decimal-variable pattern confirmed.

--- WTSAF2YR ---
Minimum: 0.0
Maximum: 521033.371
Zeros: 397
Non-missing: 3239
Missing: 0
